# Lexo — hợp nhất LoRA chọn Điều và chuyển GGUF Q4_K_M (cổng G2)

**Cài đặt notebook:** **CPU** (không cần GPU), **Internet On**.
**Input:** Output của notebook `lexo_hoc_chon_dieu` (thư mục `<TEN>/` hoặc `<TEN>.zip`).
**Chạy:** Save Version → Save & Run All. Các bước: tải Qwen3-8B đúng revision → hợp nhất ở fp16 trên CPU → `convert_hf_to_gguf.py`
ra f16 → xóa bản hợp nhất → dựng `llama-quantize` → Q4_K_M → kiểm G2 (kiến trúc qwen3, 399 tensor, có chat template).
**Output:** `<ten_ollama>-q4_k_m.gguf` (~5 GB), `merge_info.json`, bản sao `thong_tin.json` của adapter. Gắn output này vào notebook đo.

In [ ]:
TEN = "lexo-qwen3-8b-chon-dieu-v1"       # trùng TEN của notebook học
LLAMA_CPP_COMMIT = ""                    # trống = bản mới nhất; mã commit thật được ghi vào merge_info.json
PHIEN_BAN = {"transformers": "5.20.1", "peft": "0.21.2", "accelerate": "1.15.0"}
TEN_OLLAMA = "qwen3-8b-lexo-" + TEN.removeprefix("lexo-qwen3-8b-")

In [ ]:
# ---- Hàm dùng chung (không cần sửa) ----
import collections, fnmatch, hashlib, json, math, os, re, shutil, subprocess, sys, time, zipfile
from collections import Counter
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
TAM = Path("/kaggle/tmp") if Path("/kaggle/tmp").is_dir() else Path("/tmp")   # thư mục tạm lớn, không vào Output


def sha256(p, khoi=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(khoi), b""):
            h.update(b)
    return h.hexdigest()


def tim(mau, goc=INPUT):
    """Mọi file có tên khớp mẫu (không phân biệt hoa thường) dưới goc, đi theo cả symlink."""
    ra = []
    for d, _, ten in os.walk(goc, followlinks=True):
        ra += [Path(d) / t for t in ten if fnmatch.fnmatch(t.lower(), mau.lower())]
    return sorted(ra)


def tim_goi(dau, mau_zip):
    """Thư mục gói (chứa file đánh dấu). Kaggle thường tự giải nén zip; nếu không thì giải nén vào thư mục tạm."""
    thay = tim(dau)
    if not thay:
        for z in tim(mau_zip):
            dich = TAM / "giai_nen" / z.stem
            with zipfile.ZipFile(z) as f:
                f.extractall(dich)
            thay = tim(dau, dich)
            break
    if len(thay) != 1:
        raise SystemExit(f"Cần đúng một gói có {dau} trong /kaggle/input, thấy {len(thay)}: {thay}")
    return thay[0].parent


def kiem_sha256sums(goc):
    r = subprocess.run(["sha256sum", "-c", "--quiet", "SHA256SUMS"], cwd=goc, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("SHA256SUMS SAI, dừng lại (tải lại gói):\n" + r.stdout[-3000:] + r.stderr[-1000:])
    n = len([x for x in (goc / "SHA256SUMS").read_text(encoding="utf-8").splitlines() if x.strip()])
    print(f"SHA256SUMS: OK cả {n} file ({goc})")


def chay(lenh, cwd=None, env=None, duoi=None):
    """Chạy lệnh và in đầu ra từng dòng (đầu ra của subprocess thường không hiện trong ô). duoi=N: chỉ in N dòng cuối.
    Lệnh lỗi thì dừng ô."""
    lenh = [str(x) for x in lenh]
    print("$", " ".join(lenh), flush=True)
    p = subprocess.Popen(lenh, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace")
    cuoi = collections.deque(maxlen=duoi or 1)
    for dong in p.stdout:
        if duoi:
            cuoi.append(dong)
        else:
            print(dong, end="", flush=True)
    if duoi:
        print("".join(cuoi), end="", flush=True)
    if p.wait():
        raise RuntimeError(f"lệnh thoát mã {p.returncode}: {' '.join(lenh[:4])} …")


def doc_jsonl(p):
    p = Path(p)
    return [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()] if p.exists() else []

## 1. Tìm adapter, kiểm SHA256 theo `thong_tin.json`

In [ ]:
def tim_adapter(goc):
    ra = []
    for p in tim("thong_tin.json", goc):
        try:
            ok = json.loads(p.read_text(encoding="utf-8")).get("ten") == TEN
        except ValueError:
            ok = False
        if ok and (p.parent / "adapter_model.safetensors").exists():
            ra.append(p.parent)
    return ra


ung = tim_adapter(INPUT)
if not ung:
    for z in tim(f"{TEN}.zip"):
        with zipfile.ZipFile(z) as f:
            f.extractall(TAM / "adapter")
        ung = tim_adapter(TAM / "adapter")
        break
if len(ung) != 1:
    raise SystemExit(f"Cần đúng một adapter {TEN} trong /kaggle/input, thấy {len(ung)}: {ung}")
AD = ung[0]
TT = json.loads((AD / "thong_tin.json").read_text(encoding="utf-8"))
assert sha256(AD / "adapter_model.safetensors") == TT["sha256_adapter"], "SHA256 adapter không khớp thong_tin.json"
REVISION = TT["revision"]
print(f"Adapter {AD}: bước {TT['chon']['buoc']} (val loss {TT['chon']['val_loss']:.4f}), G1 {'đạt' if TT['g1']['dat'] else 'KHÔNG ĐẠT'}")
if not TT["g1"]["dat"]:
    raise SystemExit("G1 của adapter chưa đạt: không gộp. Xem thong_tin.json ở chat gốc.")
print(shutil.disk_usage(TAM))

## 2. Hợp nhất ở fp16 trên CPU

In [ ]:
chay([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k, v in PHIEN_BAN.items()]], duoi=5)
import gc, platform
import peft, torch, transformers
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM

BAN = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__}
print(BAN)
GOC_DIR, GOP_DIR = TAM / "qwen3-8b", TAM / "gop"
t0 = time.time()
snapshot_download(TT["mo_hinh_goc"], revision=REVISION, local_dir=GOC_DIR, allow_patterns=["*.json", "*.safetensors", "*.txt"])
base = AutoModelForCausalLM.from_pretrained(GOC_DIR, dtype=torch.float16, low_cpu_mem_usage=True)
gop = PeftModel.from_pretrained(base, AD).merge_and_unload()
gop.save_pretrained(GOP_DIR, safe_serialization=True, max_shard_size="4GB")
for f in GOC_DIR.iterdir():                      # tokenizer gốc nguyên bản (chat template nằm trong tokenizer_config.json)
    if f.name.startswith("tokenizer") or f.name in ("vocab.json", "merges.txt"):
        shutil.copy2(f, GOP_DIR / f.name)
del gop, base
gc.collect()
shutil.rmtree(GOC_DIR)
GIAY = {"hop_nhat": round(time.time() - t0)}
print(f"Hợp nhất xong sau {GIAY['hop_nhat'] / 60:.1f} phút;", shutil.disk_usage(TAM))

## 3. GGUF f16 → Q4_K_M bằng llama.cpp

In [ ]:
LC = TAM / "llama.cpp"
if not LC.exists():
    if LLAMA_CPP_COMMIT:
        chay(["git", "clone", "https://github.com/ggml-org/llama.cpp", LC], duoi=2)
        chay(["git", "-C", LC, "checkout", LLAMA_CPP_COMMIT], duoi=2)
    else:
        chay(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LC], duoi=2)
COMMIT = subprocess.run(["git", "-C", str(LC), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("llama.cpp", COMMIT)
chay([sys.executable, "-m", "pip", "install", "-q", str(LC / "gguf-py"), "sentencepiece", "protobuf"], duoi=5)
F16 = TAM / f"{TEN_OLLAMA}-f16.gguf"
t0 = time.time()
chay([sys.executable, LC / "convert_hf_to_gguf.py", GOP_DIR, "--outtype", "f16", "--outfile", F16], duoi=15)
shutil.rmtree(GOP_DIR)
GIAY["chuyen_f16"] = round(time.time() - t0)
if not shutil.which("cmake"):
    chay([sys.executable, "-m", "pip", "install", "-q", "cmake"], duoi=2)
t0 = time.time()
chay(["cmake", "-S", LC, "-B", LC / "build", "-DGGML_NATIVE=OFF", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"], duoi=5)
chay(["cmake", "--build", LC / "build", "--target", "llama-quantize", "-j", os.cpu_count() or 4], duoi=3)
LQ = next(p for p in (LC / "build" / "bin").glob("llama-quantize*"))
Q4 = WORK / f"{TEN_OLLAMA}-q4_k_m.gguf"
chay([LQ, F16, Q4, "Q4_K_M"], duoi=8)
F16.unlink()
GIAY["luong_tu"] = round(time.time() - t0)
print(Q4, f"{Q4.stat().st_size / 2**30:.2f} GiB")

## 4. G2 và `merge_info.json`

In [ ]:
sys.path.insert(0, str(LC / "gguf-py"))
from gguf import GGUFReader

r = GGUFReader(str(Q4))


def gia_tri(ten):
    f = r.fields.get(ten)
    if f is None:
        return None
    try:
        return f.contents()
    except AttributeError:
        v = f.parts[f.data[0]]
        return bytes(v).decode("utf-8") if v.dtype.kind == "u" and v.dtype.itemsize == 1 else v.tolist()


G2 = {"kien_truc": gia_tri("general.architecture"), "so_tensor": len(r.tensors), "co_chat_template": bool(gia_tri("tokenizer.chat_template")),
      "file_type": gia_tri("general.file_type")}
G2["dat"] = G2["kien_truc"] == "qwen3" and G2["so_tensor"] == 399 and G2["co_chat_template"]
print("G2:", G2, "→", "ĐẠT" if G2["dat"] else "KHÔNG ĐẠT")
info = {"ten": TEN, "ten_ollama": TEN_OLLAMA, "gguf": Q4.name, "kich_thuoc": Q4.stat().st_size, "sha256": sha256(Q4), "luong_tu": "Q4_K_M",
        "adapter": {"sha256": TT["sha256_adapter"], "chon": TT["chon"], "phien_ban_tin_nhan": TT["phien_ban_tin_nhan"]},
        "mo_hinh_goc": TT["mo_hinh_goc"], "revision": REVISION, "hop_nhat": "fp16 trên CPU (PeftModel.merge_and_unload)",
        "llama_cpp_commit": COMMIT, "g2": G2, "phien_ban_thu_vien": BAN, "giay": GIAY, "luc": time.strftime("%Y-%m-%d %H:%M:%S")}
(WORK / "merge_info.json").write_text(json.dumps(info, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
shutil.copy2(AD / "thong_tin.json", WORK / f"thong_tin_{TEN}.json")
print(f"sha256 {info['sha256']}\nĐã ghi {Q4.name} và merge_info.json. Tiếp: gắn Output này vào lexo_do_ban_an, bật CHAY_LORA_DEV.")